# Entrega 3 — Análisis Espacial
**Proyecto:** Patrones espaciales y temporales de delitos de alto impacto — CDMX (2019–2024)

Este notebook continúa desde la limpieza ya hecha en el archivo "H3_Procesamiento_de_Datos.ipynb".  
Carga los archivos exportados: `resultados/demografico.gpkg`, `resultados/incidencias.gpkg` y `resultados/conteo_anual.gpkg`

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import mapclassify
import warnings
warnings.filterwarnings('ignore')

import libpysal
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local
import statsmodels.api as sm

from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW

sns.set_theme(style='whitegrid')
print('Librerías cargadas.')

## 1. Carga de datos

In [ ]:
gdf          = gpd.read_file('resultados/demografico.gpkg') 
incidencias  = gpd.read_file('resultados/incidencias.gpkg') 
conteo_anual = gpd.read_file('resultados/conteo_anual.gpkg')

print(f'Colonias cargadas:   {len(gdf)}')
print(f'Delitos (puntos):    {len(incidencias)}')
print(f'Conteo anual:        {len(conteo_anual)}')
print(f'CRS colonias:        {gdf.crs}')
print(f'\nColumnas colonias:   {gdf.columns.tolist()}')
print(f'\nTipos de delito únicos: {incidencias["categoria_delito"].nunique()}')
incidencias['categoria_delito'].value_counts().head(10)

A utilizar en las secciones 5 y 6:


In [ ]:
dens_por_anio = (
    conteo_anual
    .groupby(['cve_col', 'anio_hecho'])['n_delitos']
    .sum()
    .reset_index()
)

dens_por_anio = dens_por_anio.merge(
    gdf[['cve_col', 'area_km2', 'geometry']],
    on='cve_col', how='left'
)

dens_por_anio['dens_delitos_km2'] = dens_por_anio['n_delitos'] / dens_por_anio['area_km2']
dens_por_anio['log_dens_delitos'] = np.log1p(dens_por_anio['dens_delitos_km2'])
dens_por_anio = gpd.GeoDataFrame(dens_por_anio, geometry='geometry', crs=gdf.crs)

anios = sorted(dens_por_anio['anio_hecho'].unique())
print(f"Años disponibles: {anios}")
print(f"Filas dataset anual: {len(dens_por_anio)}")

## 2 Unidad de análisis: colonia

El analisis de realiza a nivel de colonia, unidad territorial simil a un barrio, villa, vecindario. Hay tres justificaciones:

1. Granularidad adecuada: las 1,814 colonias de la CDMX permiten detectar variacion espacial intra alcaldia, que se perderia si se agregara a nivel de alcaldia, ademas escala demasiado para identficar micropatrones como lo hace Chainey.

2. Compatibilidad de fuentes: tanto los datos de incidencia como los demograficos estan disponibles con identificador de colonia lo que permite una union directa sin necesidad de interpolacion espacial.

3. Coherencia con la literatura: Vilalta & Fondevila (2022) trabajan a nivel de cuadrante policial en la CDMX, unidad que escala comparable a la colonia. Usar colonias permite replicar su enfoque.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Mapa izquierdo: colonias ---
gdf.plot(
    ax=axes[0],
    facecolor='#E8E6F0',
    edgecolor='#534AB7',
    linewidth=0.3
)
axes[0].set_title(
    f'Unidad de análisis: colonia\n({len(gdf):,} colonias válidas)',
    fontsize=13
)
axes[0].set_axis_off()

centroides = gdf.dissolve(by='alcaldia').centroid
n_por_alcaldia = gdf.groupby('alcaldia').size()
for alcaldia, punto in centroides.items():
    n = n_por_alcaldia.get(alcaldia, 0)
    axes[0].annotate(
        f'{n}',
        xy=(punto.x, punto.y),
        fontsize=7,
        ha='center',
        color='#26215C',
        fontweight='bold'
    )

# --- Mapa derecho: alcaldías (para comparar escala) ---
alcaldias = gdf.dissolve(by='alcaldia').reset_index()
alcaldias.plot(
    ax=axes[1],
    facecolor='#FAC775',
    edgecolor='#633806',
    linewidth=0.8
)
axes[1].set_title(
    f'Otra alternativa alcaldía\n({len(alcaldias)} unidades)',
    fontsize=15
)
axes[1].set_axis_off()

# Nombres de alcaldías
for _, row in alcaldias.iterrows():
    punto = row.geometry.centroid
    axes[1].annotate(
        row['alcaldia'].title(),
        xy=(punto.x, punto.y),
        fontsize=6.5,
        ha='center',
        color='#412402',
        fontweight='bold'
    )

plt.suptitle(
    'Justificación de la unidad de análisis\nColonia vs Alcaldía — CDMX',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig('resultados/00_unidad_analisis.png', dpi=150, bbox_inches='tight')
plt.savefig('resultados/00_unidad_analisis.svg', format='svg', bbox_inches='tight')
plt.show()

print(f'Colonias: {len(gdf):,} unidades')
print(f'Alcaldías: {len(alcaldias)} unidades')
print(f'Ratio: {len(gdf)/len(alcaldias):.0f}x más resolución con colonias')

## 3. Mapas coropléticos — total y por categoria

Sub Pregunta 1: ¿Las distintas categorías de delito se concentran en las mismas colonias o cada una tiene su propio patrón espacial?

Se visualiza la distribucion espacial de la densidad de delitos por $km^2$ usando clasificacion Fisher-Jenks con k=5. Se generan dos mapas: uno con la densidad total (todas las categorías agregadas) y otro comparando las 4 categorías más frecuentes, para responder si distintas categorías presentan patrones espaciales distintas.

In [ ]:
np.random.seed(12345)  # Semilla 
jc5 = mapclassify.JenksCaspall(gdf["log_dens_delitos"], k=5)
jc5

In [ ]:
# ── Mapa 1: Tasa total por colonia ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))

gdf.plot(
    column='log_dens_delitos',
    scheme='FisherJenks',
    k=5,
    cmap='OrRd',
    legend=True,
    ax=ax,
    edgecolor='black',
    linewidth=0.1,
    legend_kwds={
        'title': 'Densidad delitos por km² por colonia.',
        'fontsize': 9,
        'title_fontsize': 10
    }
)

ax.set_title(
    'Densidad de delitos de alto impacto por colonia\nCDMX 2019–2024 (clasificación Jenks)',
    fontsize=14, pad=15
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('resultados/01_mapa_tasa_total.png', dpi=150, bbox_inches='tight')
plt.savefig('resultados/01_mapa_tasa_total.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
# ── Mapa 2: Comparación entre tipos de delito ─────────────────────────────────
# Toma los 4 tipos más frecuentes
top4_cat = incidencias['categoria_delito'].value_counts().head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

cmaps = ['YlOrBr', 'Blues', 'Greens', 'Purples']

for i, categoria in enumerate(top4_cat):
    # Nombre de columna tal como quedó en el pivot (prefijo n_ y guiones bajos)
    col_n    = f'n_{categoria[:30].strip().replace(" ", "_")}'
    col_dens = f'dens_{categoria[:20].strip().replace(" ", "_")}_km2'

    # Calcular densidad por km^2 para esta categoría si no existe
    if col_dens not in gdf.columns:
        if col_n in gdf.columns:
            gdf[col_dens] = np.where(
                gdf['area_km2'] > 0,
                gdf[col_n] / gdf['area_km2'],
                np.nan
            )
            gdf[col_dens] = np.log1p(gdf[col_dens])  # log para normalizar
        else:
            print(f'⚠ Columna no encontrada: {col_n}')
            continue

    gdf.plot(
        column=col_dens,
        scheme='FisherJenks',
        k=5,
        cmap=cmaps[i],
        legend=True,
        ax=axes[i],
        edgecolor='black',
        linewidth=0.05,
        legend_kwds={'title': 'log(delitos/km^2)', 'fontsize': 7}
    )
    axes[i].set_title(f'{categoria[:55]}', fontsize=10, pad=8)
    axes[i].set_axis_off()

plt.suptitle(
    '¿Los hotspots varían según la categoría de delito?\nDensidad por km^2 (log) por colonia — CDMX 2019–2024',
    fontsize=15, y=1.01
)
plt.tight_layout()
plt.savefig('resultados/02_mapas_por_categoria.png', dpi=150, bbox_inches='tight')
plt.savefig('resultados/02_mapas_por_categoria.svg', format='svg', bbox_inches='tight')
plt.show()

Al separar por categoría, cada delito muestra su propia geografía:

- El robo a transeúnte se concentra en el centro norte.

- El robo de vehículo se concentra en el  centro.

- El robo a negocio es mas difuso, pero se concentra en el norte.

- El robo repartidor se concentra mas en la periferia.

El centro aparece en dos de las cuatros categorias, pero cada uno tiene su propio patrón, confirmando que cada categoría tiene su propia distribucion espacial.

## 4. Lorenz / Gini

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4. Concentración de delitos — Curva de Lorenz y Gini
# ══════════════════════════════════════════════════════════════════

def lorenz_gini(valores):
    """Calcula la curva de Lorenz y el coeficiente de Gini."""
    v = np.sort(valores[valores > 0])
    n = len(v)
    cumsum = np.cumsum(v)
    lorenz_x = np.arange(1, n + 1) / n
    lorenz_y = cumsum / cumsum[-1]
    gini = 1 - 2 * np.trapezoid(lorenz_y, lorenz_x)
    return lorenz_x, lorenz_y, gini

fig, ax = plt.subplots(figsize=(8, 7))

# Línea de igualdad perfecta
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2,
        label='Distribución uniforme', alpha=0.6)

colores = ['#D62728', '#1F77B4', '#2CA02C', '#FF7F0E']

# Curva total
x, y, g = lorenz_gini(gdf['n_delitos'].values)
ax.plot(x, y, color='black', linewidth=2.5,
        label=f'Total (Gini = {g:.3f})')

# Curva por top 4 categorías
for i, cat in enumerate(top4_cat[:4]):
    col_n = f'n_{cat[:30].strip().replace(" ", "_")}'
    if col_n in gdf.columns:
        x, y, g = lorenz_gini(gdf[col_n].values)
        ax.plot(x, y, color=colores[i], linewidth=1.5, alpha=0.85,
                label=f'{cat[:35]} (Gini={g:.3f})')
    else:
        print(f'⚠ Columna no encontrada: {col_n}')

# Líneas de referencia
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
ax.axhline(0.25, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

ax.set_xlabel('Proporción acumulada de colonias (de menor a mayor)', fontsize=11)
ax.set_ylabel('Proporción acumulada de delitos', fontsize=11)
ax.set_title(
    'Curva de Lorenz — Concentración de delitos por colonia\n'
    'CDMX 2019–2024',
    fontsize=13
)
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('resultados/04_lorenz_gini.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/04_lorenz_gini.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Resumen numérico ──────────────────────────────────────────────
vals    = np.sort(gdf['n_delitos'].values)
cumsum  = np.cumsum(vals) / vals.sum()
idx_50  = np.searchsorted(cumsum, 0.5)
pct_col = (1 - idx_50 / len(vals)) * 100

print(f'El {pct_col:.1f}% de las colonias concentra el 50% de los delitos.')

Todos los Gini superan 0.52, confirmando alta concentración. El 15.7% de las colonias concentra el 50% de los delitos de alto impacto.

El robo a transeúnte es el que tiene mayor valor (Gini=0.579) y el robo a vehículo es el menor (Gini=0.523), considerando el top 4 categorías.

Chainey et al. (2019) mencionan que entre un 2% y 6% de los micro-lugares concentran el 50% de los crímenes en ciudades de latinoamerica. El 15.7% es mayor porque la colonia es una unidad mas grande que un micro-lugar.

## 5. Distribución temporal por tipo de delito

Sub Pregunta 2:	¿Cómo varia la ocurrencia de delitos según la hora del día, y esta variación depende de la categoría del delito?

Se analiza cómo varía la frecuencia de delitos segun la hora del dia, y si ese patrón difere entre categorías. Se normaliza por categoría para comparar patrones.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ANÁLISIS TEMPORAL 1: Patrón horario por categoría de delito
# ══════════════════════════════════════════════════════════════════
top5_cat = incidencias['categoria_delito'].value_counts().head(5).index.tolist()
incidencias_top = incidencias[incidencias['categoria_delito'].isin(top5_cat)].copy()

pivot_hora = (
    incidencias_top
    .groupby(['hora_hecho', 'categoria_delito'])
    .size()
    .reset_index(name='n')
    .pivot_table(index='hora_hecho', columns='categoria_delito', values='n', fill_value=0)
)

pivot_hora_norm = pivot_hora.div(pivot_hora.sum(axis=0), axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sns.heatmap(
    pivot_hora_norm.T,
    cmap='YlOrRd',
    ax=axes[0],
    cbar_kws={'label': 'Proporción horaria'},
    linewidths=0.3
)
axes[0].set_title('Patrón horario por categoría de delito\n(normalizado por categoría)', fontsize=12)
axes[0].set_xlabel('Hora del día')
axes[0].set_ylabel('')
axes[0].set_yticklabels([t[:40] for t in pivot_hora_norm.columns], fontsize=8)

for cat in top5_cat:
    if cat in pivot_hora_norm.columns:
        axes[1].plot(pivot_hora_norm.index, pivot_hora_norm[cat],
                     marker='o', markersize=3, linewidth=1.5, label=cat[:40])
axes[1].set_title('Curvas horarias por categoría de delito', fontsize=12)
axes[1].set_xlabel('Hora del día (0–23 hrs)')
axes[1].set_ylabel('Proporción de delitos de la categoría')
axes[1].legend(fontsize=8, loc='upper left')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('resultados/03a_patron_horario.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/03a_patron_horario.png', dpi=150, bbox_inches='tight')
plt.show()

El heatmap y las curvas horarios muestran que las categorías no siguen patrones iguales, confirmando que la hora del delito depende del tipo:

- Robo a repartidor (rojo): peak concentrado entre las 10 y 14 horas, hoaraio de maxima actividad de enregas. Es el patrón más diferente.

- Robo a negocio (verde): Sube prograsivamente durante el día con peak entre las 20 y 21 horas, justo en el horario de cierre de estas.

- Robo a transeúnte (azul): un patron mas distribuido a lo largo del dia, con un aumento sostenido desde las 4 hasta las 15.

La variación horaria entre categorías confirma la subpregunta 2: el cuándo del delito depende del tipo de delito.

## 6. Autocorrelación espacial y hotpots
Sub Pregunta 3: ¿Existen hotpots delictivos concentrados en ciertas colonias?

### 5.1 Autocorrelación espacial global — Moran's I

Verifica si existe clustering espacial en la tasa de delitos.

In [ ]:
# ── 5.1 Matriz de pesos Queen ─────────────────────────────────────
w = Queen.from_dataframe(gdf, silence_warnings=True)
w.transform = 'r'

print(f'Colonias en la matriz: {w.n}')
print(f'Vecinos promedio:      {w.mean_neighbors:.1f}')
print(f'Islas (sin vecinos):   {len(w.islands)}')

In [ ]:
# ── 5.2 Moran I global ────────────────────────────────────────────
y_arr = gdf['log_dens_delitos'].values
moran_global = Moran(y_arr, w)

print('=' * 45)
print(f"  Moran's I  = {moran_global.I:.4f}")
print(f"  p-valor    = {moran_global.p_sim:.4f}")
print('=' * 45)

# Diagrama de Moran
lag_y = libpysal.weights.lag_spatial(w, y_arr)
y_std   = (y_arr - y_arr.mean()) / y_arr.std()
lag_std = (lag_y - lag_y.mean()) / lag_y.std()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_std, lag_std, alpha=0.25, s=6, color='#534AB7')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.plot(sorted(y_std),
        [moran_global.I * xi for xi in sorted(y_std)],
        color='#D85A30', linewidth=2,
        label=f"Moran's I = {moran_global.I:.3f}  (p = {moran_global.p_sim:.4f})")

for txt, xy in [('HH',(1.5,1.5)),('LL',(-1.5,-1.5)),('HL',(1.5,-1.5)),('LH',(-1.5,1.5))]:
    ax.text(*xy, txt, fontsize=11, color='gray', ha='center', alpha=0.7)

ax.set_xlabel('log(densidad delitos/km^2) estandarizada')
ax.set_ylabel('Lag espacial estandarizado')
ax.set_title("Diagrama de Moran — Densidad total de delitos", fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('resultados/04a_moran_scatter_global.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/04a_moran_scatter_global.png', dpi=150, bbox_inches='tight')
plt.show()

Resultado: I ≈ 0.383 (p = 0,001) → autocorrelación espacial positiva y significativa. Esto significa que las colonias con alta densidad de delitos tienden a estar rodeadas de otras colonias con alta densidad. El patron no es aleatorio.

In [ ]:
# ── 5.3 Moran I por categoría de delito ───────────────────────────
print("Moran's I por categoría de delito:")
print(f'{"Categoría":<45}  {"I":>7}  {"p":>7}')
print('-' * 62)

resultados_moran_cat = []
for cat in top4_cat:
    col_n    = f'n_{cat[:30].strip().replace(" ", "_")}'

    if col_n in gdf.columns:
        # Calcular log-densidad por km² para esta categoría
        dens = np.where(gdf['area_km2'] > 0, gdf[col_n] / gdf['area_km2'],
            np.nan)
        y_cat = np.log1p(np.nan_to_num(dens, nan=0))
        m = Moran(y_cat, w)
        sig = '✓' if m.p_sim < 0.05 else ' '
        print(f'{sig} {cat[:43]:<43}  {m.I:>7.4f}  {m.p_sim:>7.4f}')
        resultados_moran_cat.append({'categoria': cat, 'I': m.I, 'p': m.p_sim})
    else:
        print(f'  ⚠ columna no encontrada: {col_n}')


Las cuatro categorías muestran autocorrelación positiva y significativa.

Los cuatro tipos muestran autocorrelacion positiva y significativa. El robo a transeunte con y sin violencia tiene el I mas alto (0.42), un poco mas alto que el I global, lo que indica una concentracion espacial mucho mas fuerte que el promedio.

In [ ]:
# ── 5.4 Moran I por año (tabla) ───────────────────────────────────
resultados_moran_anual = []

for anio in anios:
    gdf_anio = dens_por_anio[dens_por_anio['anio_hecho'] == anio].copy()
    gdf_anio = gdf_anio.reset_index(drop=True)

    w_anio = Queen.from_dataframe(gdf_anio, silence_warnings=True)
    w_anio.transform = 'r'

    y_anio = gdf_anio['log_dens_delitos'].values
    moran  = Moran(y_anio, w_anio)

    resultados_moran_anual.append({
        'anio':    anio,
        'moran_I': moran.I,
        'p_valor': moran.p_sim,
        'sig':     '✓' if moran.p_sim < 0.05 else '✗'
    })

df_moran_anual = pd.DataFrame(resultados_moran_anual)

# Tabla formateada
print("\nMoran's I por año:")
print(f"{'Año':<6}  {'Moran I':>8}  {'p-valor':>8}  {'Sig.':>5}")
print('-' * 35)
for _, row in df_moran_anual.iterrows():
    nota = '*' if row['anio'] == 2024 else ''
    print(f"{int(row['anio']):<6}{nota}  {row['moran_I']:>8.4f}  {row['p_valor']:>8.4f}  {row['sig']:>5}")
print("\n* datos hasta septiembre 2024")

El I de Moran es significativo y positivo en todos los año. Sin embargo, con una tendencia descendiente, lo que sugiere que la concentracion espacial se ha debilitado. La caída mas brica ocurren en 2020 (pandemia), con una leve recuperacion, pero sin volver a los valores de pre-pandemia

In [ ]:
# ── 5.5 Diagramas de Moran por año (2019, 2021, 2024) ────────────
anios_seleccionados = [2019, 2021, 2024]
titulos_anio = {
    2019: "2019 (Pre-pandemia)",
    2021: "2021 (Pandemia)",
    2024: "2024* (Post-pandemia)"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, anio in enumerate(anios_seleccionados):
    ax = axes[i]
    gdf_a = dens_por_anio[dens_por_anio['anio_hecho'] == anio].copy().reset_index(drop=True)
    w_a = Queen.from_dataframe(gdf_a, silence_warnings=True)
    w_a.transform = 'r'
    y_a  = gdf_a['log_dens_delitos'].values
    moran_a   = Moran(y_a, w_a)

    # Estandarizar
    lag_y   = libpysal.weights.lag_spatial(w_a, y_a)
    y_std   = (y_a - y_a.mean()) / y_a.std()
    lag_std = (lag_y  - lag_y.mean())  / lag_y.std()

    # Scatter
    ax.scatter(y_std, lag_std, alpha=0.2, s=5, color='#534AB7')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')

    # Línea de regresión
    ax.plot(sorted(y_std),
            [moran_a.I * xi for xi in sorted(y_std)],
            color='#D85A30', linewidth=2,
            label=f"I = {moran_a.I:.3f}  (p = {moran_a.p_sim:.4f})")

    # Etiquetas cuadrantes
    for txt, xy in [('HH',(1.5,1.5)),('LL',(-1.5,-1.5)),
                    ('HL',(1.5,-1.5)),('LH',(-1.5,1.5))]:
        ax.text(*xy, txt, fontsize=11, color='gray', ha='center', alpha=0.7)

    ax.set_xlabel('log(dens. delitos/km²) estand.', fontsize=9)
    ax.set_ylabel('Lag espacial estand.' if i == 0 else '', fontsize=9)
    ax.set_title(titulos_anio[anio], fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle("Diagrama de Moran por año — CDMX", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('resultados/04b_moran_scatter_anios.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/04b_moran_scatter_anios.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 LISA — Mapa de hotspots y coldspots

El analisis LISA identifica colonias con clsutering espacial significativo. Responde directamente a la subpregunta 2 sobre la existencia de hotpots.

In [ ]:
# LISA sobre tasa total
moran_loc = Moran_Local(y_arr, w, seed=42)

sig = moran_loc.p_sim < 0.05
q   = moran_loc.q

gdf['lisa_cat'] = 'No significativo'
gdf.loc[sig & (q == 1), 'lisa_cat'] = 'HH (hotspot)'
gdf.loc[sig & (q == 3), 'lisa_cat'] = 'LL (coldspot)'
gdf.loc[sig & (q == 2), 'lisa_cat'] = 'LH'
gdf.loc[sig & (q == 4), 'lisa_cat'] = 'HL'

print(gdf['lisa_cat'].value_counts())

# Mapa LISA
color_map = {
    'HH (hotspot)':    '#D62728',
    'LL (coldspot)':   '#1F77B4',
    'LH':              '#AEC7E8',
    'HL':              '#FFBB78',
    'No significativo':'#D3D3D3',
}

fig, ax = plt.subplots(figsize=(10, 10))

for cat, color in color_map.items():
    subset = gdf[gdf['lisa_cat'] == cat]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, edgecolor='white', linewidth=0.2, label=cat)

ax.set_title(
    f'Clusters LISA — Densidad de delitos de alto impacto por colonia\n'
    f"Moran's I = {moran_global.I:.3f}, p = {moran_global.p_sim:.4f}",
    fontsize=14, pad=15
)
ax.legend(title='Tipo de cluster', loc='lower right', fontsize=10)
ax.set_axis_off()

plt.tight_layout()
plt.savefig('resultados/05a_mapa_lisa.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/05a_mapa_lisa.png', dpi=150, bbox_inches='tight')
plt.show()

De las 1,814 colonias, 383 son hotspots (HH) y 158 coldspots (LL), ambos significativos. Los hotspots se concentran en el centro y norte de la ciudad, mientras que los coldspots estan en la periferia sur y oeste. Solo 29 colonias muestran un cluster espacial tipo LH o HL.

In [ ]:
# ── LISA para 3 años seleccionados ───────────────────────────────
anios_seleccionados = [2019, 2021, 2024]
titulos_anio = {
    2019: "2019\n(Pre-pandemia)",
    2021: "2021\n(Pandemia)",
    2024: "2024*\n(Post-pandemia)"
}

color_map = {
    'HH (hotspot)':     '#D62728',
    'LL (coldspot)':    '#1F77B4',
    'LH':               '#AEC7E8',
    'HL':               '#FFBB78',
    'No significativo': '#D3D3D3',
}

fig, axes = plt.subplots(1, 3, figsize=(21, 8))

for i, anio in enumerate(anios_seleccionados):
    ax = axes[i]

    # Datos de ese año
    gdf_anio = dens_por_anio[dens_por_anio['anio_hecho'] == anio].copy()
    gdf_anio = gdf_anio.reset_index(drop=True)

    # Matriz de pesos
    w_anio = Queen.from_dataframe(gdf_anio, silence_warnings=True)
    w_anio.transform = 'r'

    # LISA
    y_anio    = gdf_anio['log_dens_delitos'].values
    moran_a   = Moran(y_anio, w_anio)
    moran_loc = Moran_Local(y_anio, w_anio, seed=42)

    sig = moran_loc.p_sim < 0.05
    q   = moran_loc.q

    gdf_anio['lisa_cat'] = 'No significativo'
    gdf_anio.loc[sig & (q == 1), 'lisa_cat'] = 'HH (hotspot)'
    gdf_anio.loc[sig & (q == 3), 'lisa_cat'] = 'LL (coldspot)'
    gdf_anio.loc[sig & (q == 2), 'lisa_cat'] = 'LH'
    gdf_anio.loc[sig & (q == 4), 'lisa_cat'] = 'HL'

    # Mapa
    for cat, color in color_map.items():
        subset = gdf_anio[gdf_anio['lisa_cat'] == cat]
        if len(subset) > 0:
            subset.plot(ax=ax, color=color, edgecolor='white',
                       linewidth=0.2, label=cat)

    # Conteo de HH y LL
    n_hh = (gdf_anio['lisa_cat'] == 'HH (hotspot)').sum()
    n_ll = (gdf_anio['lisa_cat'] == 'LL (coldspot)').sum()

    ax.set_title(
        f"{titulos_anio[anio]}\n"
        f"I = {moran_a.I:.3f}  |  HH={n_hh}  LL={n_ll}",
        fontsize=11
    )
    ax.set_axis_off()

    # Leyenda solo en el último mapa
    if i == 2:
        ax.legend(title='Tipo de cluster', loc='lower right', fontsize=8)

plt.suptitle(
    'Evolución de clusters LISA — Densidad de delitos por colonia\n'
    'CDMX 2019, 2021 y 2024  (* datos hasta septiembre)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('resultados/05b_lisa_por_anio.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/05b_lisa_por_anio.png', dpi=150, bbox_inches='tight')
plt.show()

La evolución temporal muestra una reducción sostenida de los HH de 371 colonias en 2019 a 218 en 2024. Con los LL pasa lo mismo se reducen 170 a 126. El núcleo de hotspots en el centro se mantiene estable en los tres años, pero su expansion se reduce en el sur-este progresivamente, la pandemia (2021) marca el inicio de reducción, que no se revierte en 2024. Para los coldspots si se mantienen las mismas zonas.

## 7. Implementación OLS y GWR

Se implementa primero un OLS global como modelo base, para luego dar paso a una regresion geograficamente ponderada (GWR) siguiendo a **Vilalta & Fondevila (2022)**. La variable dependiente es la log-densidad de delitos por $km^2$ y los predictores son densidad poblacional, densidad de cámaras y densidad de aglomeraciones y el logaritmo de la densidad poblacional como variable independiente. Los puntos usados son los centroides de las colonias no los puntos de delutos individuales.

### 6.1 Correlación de variables

Se utilizan densidades en lugar de valores brutos para hacer las colonias comparables entre sí, dado que la variable dependiente también está expresada en densidad (delitos/$km^2$).

In [ ]:
# ── 6.1 Verificación de multicolinealidad ─────────────────────────
y_col  = 'log_dens_delitos'
x_cols = ['log_dens_pob', 'dens_camaras_km2', 'dens_aglom_km2']

cols_modelo = [y_col] + x_cols
df_modelo   = gdf[cols_modelo].dropna().copy()

print(f"Colonias con datos completos: {len(df_modelo)} de {len(gdf)}")
print("\nCorrelación de cada predictor con log_dens_delitos:")
print(df_modelo.corr()[y_col].drop(y_col).round(3).to_string())

print("\nCorrelaciones entre predictores (⚠ si |r| > 0.7):")
corr_X = df_modelo[x_cols].corr()
for i in range(len(x_cols)):
    for j in range(i+1, len(x_cols)):
        r = corr_X.iloc[i, j]
        flag = '⚠' if abs(r) > 0.7 else '·' if abs(r) > 0.5 else ' '
        print(f"  {flag} {x_cols[i]:25} × {x_cols[j]:25}  r = {r:.3f}")

### 6.2 OLS global + diagnóstico espacial

In [ ]:
# ── 6.2 OLS global + diagnóstico espacial ─────────────────────────
idx      = df_modelo.index
y        = df_modelo[y_col].values
X        = sm.add_constant(df_modelo[x_cols].values)
ols      = sm.OLS(y, X).fit()

# Resumen simplificado
print('=' * 50)
print("RESULTADOS OLS")
print('=' * 50)
print(f"  R²          = {ols.rsquared:.4f}")
print(f"  R² ajustado = {ols.rsquared_adj:.4f}")
print(f"  AIC         = {ols.aic:.2f}")
print(f"\n  Coeficientes:")
nombres = ['constante'] + x_cols
for nombre, coef, pval in zip(nombres, ols.params, ols.pvalues):
    sig = '✓' if pval < 0.05 else '✗'
    print(f"  {sig} {nombre:25}  β={coef:.4f}  p={pval:.4f}")
print('=' * 50)

# Moran sobre residuos
gdf_modelo          = gdf.loc[idx].copy()
gdf_modelo['ols_residuos'] = ols.resid
w_ols               = Queen.from_dataframe(gdf_modelo, silence_warnings=True)
w_ols.transform     = 'r'
moran_resid         = Moran(ols.resid, w_ols)

print('\n' + '=' * 50)
print("TEST DE MORAN SOBRE RESIDUOS OLS")
print('=' * 50)
print(f"  Moran's I  = {moran_resid.I:.4f}")
print(f"  p-valor    = {moran_resid.p_sim:.4f}")
if moran_resid.p_sim < 0.05:
    print("\n  → Residuos con autocorrelación espacial significativa.")
    print("    Esto JUSTIFICA el uso del GWR.")
else:
    print("\n  → Residuos sin autocorrelación espacial.")
print('=' * 50)

# Mapa de residuos
vabs = np.abs(gdf_modelo['ols_residuos']).quantile(0.95)
fig, ax = plt.subplots(figsize=(8, 8))
gdf_modelo.plot(
    column='ols_residuos',
    cmap='RdBu_r',
    vmin=-vabs, vmax=vabs,
    legend=True,
    ax=ax,
    edgecolor='white', linewidth=0.15,
    legend_kwds={'label': 'Residuo OLS', 'shrink': 0.7}
)
ax.set_title(
    f'Residuos del OLS\nMoran I = {moran_resid.I:.3f}  (p = {moran_resid.p_sim:.4f})\n',
    fontsize=12
)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('resultados/06b_ols_residuos.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/06b_ols_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

El OLS global con tres predictores obtiene un $R^2=0.2840$, explicando un 28% de la densidad de delitos. Con los tres coef. significativos:

- Densidad poblacional: a mayor densidad de población, mayor densidad de delitos.
- Densidad de cámaras: coeficiente positivo, que refleja causalidad inversa - las cámaras se instalan donde ya hay más delitos.
- Densidad de aglomeraciones: el efecto más pequeño en magnitud pero igualmente significativo.

El mapa de residuos muestra zonas rojas en el centro y centro norte y zonas azules en la periferia, confirmado por el test de Moran sobre residuos. Esto indica que el OLS deja estructura espacial sin explicar, justificando el salto al GWR.

### 6.3 GWR global + mapas de coeficientes

In [ ]:
gdf_proj = gdf.to_crs('EPSG:32614')
coords   = np.column_stack([
    gdf_proj.geometry.centroid.x,
    gdf_proj.geometry.centroid.y
])

y_gwr = gdf[[y_col]].values.astype(float)
X_gwr = gdf[x_cols].values.astype(float)

selector = Sel_BW(coords, y_gwr, X_gwr, kernel='gaussian', fixed=False, constant=True)
bw       = selector.search(criterion='AICc', search_method='golden_section')
print(f'Bandwidth óptimo: {bw:.0f} vecinos ({bw/len(y_gwr)*100:.1f}% de colonias)')

gwr_model   = GWR(coords, y_gwr, X_gwr, bw, kernel='gaussian', fixed=False, constant=True)
gwr_results = gwr_model.fit()

# Extraer resultados
gdf['gwr_coef_dens_pob']  = gwr_results.params[:, 1]
gdf['gwr_coef_camaras']   = gwr_results.params[:, 2]
gdf['gwr_coef_aglom']     = gwr_results.params[:, 3]
gdf['gwr_r2_local']       = gwr_results.localR2
gdf['gwr_sig_dens_pob']   = np.abs(gwr_results.tvalues[:, 1]) > 1.96
gdf['gwr_sig_camaras']    = np.abs(gwr_results.tvalues[:, 2]) > 1.96
gdf['gwr_sig_aglom']      = np.abs(gwr_results.tvalues[:, 3]) > 1.96

print(f"\nGWR R² = {gwr_results.R2:.4f}  vs  OLS R² = {ols.rsquared:.4f}")
print(f"AIC   = {gwr_results.aic:.2f}  vs  OLS AIC = {ols.aic:.2f}")
print(f"\n% colonias con efecto significativo:")
for col_sig, nombre in [('gwr_sig_dens_pob','dens_pob'),
                         ('gwr_sig_camaras', 'camaras'),
                         ('gwr_sig_aglom',   'aglom')]:
    print(f"  {nombre:15} {gdf[col_sig].mean()*100:.1f}%")

# Mapas 2×2
def plot_coef(ax, col, titulo, etiqueta):
    vmin = gdf[col].quantile(0.05)
    vmax = gdf[col].quantile(0.95)
    if vmin < 0 < vmax:
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
        cmap = 'RdBu_r'
    else:
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap = 'Reds' if vmin >= 0 else 'Blues_r'
    gdf.plot(column=col, cmap=cmap, norm=norm, legend=True, ax=ax,
             edgecolor='white', linewidth=0.15,
             legend_kwds={'label': etiqueta, 'shrink': 0.7})
    ax.set_title(titulo, fontsize=11)
    ax.set_axis_off()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

plot_coef(axes[0,0], 'gwr_coef_dens_pob',
          'Coeficiente local — Densidad poblacional', 'β densidad pob.')
plot_coef(axes[0,1], 'gwr_coef_camaras',
          'Coeficiente local — Densidad de cámaras', 'β cámaras/km²')
plot_coef(axes[1,0], 'gwr_coef_aglom',
          'Coeficiente local — Densidad de aglomeraciones', 'β aglom./km²')

gdf.plot(column='gwr_r2_local', scheme='NaturalBreaks', k=5,
         cmap='YlGn', legend=True, ax=axes[1,1],
         edgecolor='white', linewidth=0.15,
         legend_kwds={'title': 'R² local', 'fontsize': 8})
axes[1,1].set_title(
    f'R² local del GWR\n(R² global={gwr_results.R2:.3f}  vs  OLS={ols.rsquared:.3f})',
    fontsize=11)
axes[1,1].set_axis_off()

plt.suptitle('GWR — Heterogeneidad espacial de los efectos\nCDMX 2019–2024',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('resultados/06c_gwr_coeficientes.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/06c_gwr_coeficientes.png', dpi=150, bbox_inches='tight')

plt.show()

El GWR con bandwidth óptimo de 49 vecinos alcanza un $R^2=0.500$, casi el doble que el OLS (0.284), y mejora el AICc de 5396 a 4878, confirmando que los efectos varían espacialmente y que el GWR es el modelo más adecuado.

- Densidad poblacional (sig. en 50.7% de colonias): es la variable con mayor heterogeneidad espacial. El coeficiente cambia de signo: negativo en el centro y norte (donde la población residente no explica los delitos porque estos ocurren donde la gente circula, no donde vive) y positivo en la periferia sur y oriente.

- Densidad de cámaras (sig. en 82.2% de colonias): coeficiente positivo en casi toda la ciudad, consistente con causalidad inversa: las cámaras se instalan donde ya hay más delitos.

- Densidad de aglomeraciones (sig. en 97.6% de colonias): el efecto más extendido y estable. Mayor en la periferia donde la actividad económica contrasta más con el entorno, y más débil en el centro donde ya hay saturación de aglomeraciones.

- El $R^2$ local es más bajo en el centro (0.14–0.26) y más alto en la periferia (0.47–0.58), indicando que en el centro operan factores no capturados por el modelo.

### 6.4 OLS por año

In [ ]:
resultados_ols = []

print(f"{'Año':<6} {'R²':>6} {'β_pob':>8} {'β_cam':>8} {'β_aglom':>9}  Moran_resid")
print('-' * 65)

for anio in anios:
    gdf_anio = dens_por_anio[dens_por_anio['anio_hecho'] == anio].copy()
    gdf_anio = gdf_anio.reset_index(drop=True)
    gdf_anio = gdf_anio.merge(gdf[['cve_col'] + x_cols], on='cve_col', how='left')
    gdf_anio = gdf_anio.dropna(subset=['log_dens_delitos'] + x_cols).reset_index(drop=True)

    y_a = gdf_anio['log_dens_delitos'].values
    X_a = sm.add_constant(gdf_anio[x_cols].values)
    ols_a = sm.OLS(y_a, X_a).fit()

    w_a = Queen.from_dataframe(gdf_anio, silence_warnings=True)
    w_a.transform = 'r'
    mr = Moran(ols_a.resid, w_a)

    resultados_ols.append({
        'anio': anio, 'r2': ols_a.rsquared,
        'beta_pob': ols_a.params[1], 'beta_cam': ols_a.params[2],
        'beta_aglom': ols_a.params[3],
        'p_pob': ols_a.pvalues[1], 'p_cam': ols_a.pvalues[2],
        'p_aglom': ols_a.pvalues[3],
        'moran_I': mr.I, 'moran_p': mr.p_sim,
    })

    sig = '✓' if mr.p_sim < 0.05 else '✗'
    print(f"{anio:<6} {ols_a.rsquared:>6.3f} {ols_a.params[1]:>8.3f} "
          f"{ols_a.params[2]:>8.4f} {ols_a.params[3]:>9.5f}  "
          f"I={mr.I:.3f} p={mr.p_sim:.3f} {sig}")

df_ols = pd.DataFrame(resultados_ols)

# Gráfico 2×2
fig, axes = plt.subplots(2, 2, figsize=(7, 7))

# R²
axes[0,0].plot(df_ols['anio'], df_ols['r2'], marker='o',
               linewidth=2.5, markersize=8, color='#534AB7')
axes[0,0].axvspan(2019.5, 2021.5, alpha=0.12, color='gray', label='Pandemia')
axes[0,0].set_title('R² del OLS por año', fontsize=12)
axes[0,0].set_ylabel('R²')
axes[0,0].set_xticks(anios)
axes[0,0].legend(fontsize=9)
for _, row in df_ols.iterrows():
    axes[0,0].annotate(f"{row['r2']:.3f}", xy=(row['anio'], row['r2']),
                       xytext=(0,8), textcoords='offset points',
                       ha='center', fontsize=8, color='#534AB7')

# β densidad pob
sig_pob = df_ols['p_pob'] < 0.05
axes[0,1].plot(df_ols['anio'], df_ols['beta_pob'], marker='o',
               linewidth=2.5, markersize=8, color='#D85A30')
axes[0,1].scatter(df_ols.loc[~sig_pob,'anio'], df_ols.loc[~sig_pob,'beta_pob'],
                  color='gray', s=80, zorder=5, label='No significativo')
axes[0,1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0,1].axvspan(2019.5, 2021.5, alpha=0.12, color='gray')
axes[0,1].set_title('β Densidad poblacional por año', fontsize=12)
axes[0,1].set_ylabel('Coeficiente')
axes[0,1].set_xticks(anios)
axes[0,1].legend(fontsize=9)

# β cámaras
sig_cam = df_ols['p_cam'] < 0.05
axes[1,0].plot(df_ols['anio'], df_ols['beta_cam'], marker='o',
               linewidth=2.5, markersize=8, color='#26A269')
axes[1,0].scatter(df_ols.loc[~sig_cam,'anio'], df_ols.loc[~sig_cam,'beta_cam'],
                  color='gray', s=80, zorder=5, label='No significativo')
axes[1,0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1,0].axvspan(2019.5, 2021.5, alpha=0.12, color='gray')
axes[1,0].set_title('β Densidad de cámaras por año', fontsize=12)
axes[1,0].set_ylabel('Coeficiente')
axes[1,0].set_xticks(anios)
axes[1,0].legend(fontsize=9)

# β aglomeraciones
sig_aglom = df_ols['p_aglom'] < 0.05
axes[1,1].plot(df_ols['anio'], df_ols['beta_aglom'], marker='o',
               linewidth=2.5, markersize=8, color='#A347BA')
axes[1,1].scatter(df_ols.loc[~sig_aglom,'anio'], df_ols.loc[~sig_aglom,'beta_aglom'],
                  color='gray', s=80, zorder=5, label='No significativo')
axes[1,1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1,1].axvspan(2019.5, 2021.5, alpha=0.12, color='gray')
axes[1,1].set_title('β Densidad de aglomeraciones por año', fontsize=12)
axes[1,1].set_ylabel('Coeficiente')
axes[1,1].set_xticks(anios)
axes[1,1].legend(fontsize=9)

for ax in axes.flatten():
    ax.annotate('* 2024 parcial\n(ene–sept)',
                xy=(2024, ax.get_ylim()[0]),
                xytext=(-30, 15), textcoords='offset points',
                fontsize=7, color='gray')

plt.suptitle('Evolución de los coeficientes OLS por año\nCDMX 2019–2024',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('resultados/09_ols_por_anio.svg', format='svg', bbox_inches='tight')
plt.savefig('resultados/09_ols_por_anio.png', dpi=150, bbox_inches='tight')
plt.show()

El $R^2$ cae de 0.241 (2019) a 0.178 (2022), indicando que los predictores explican cada vez menos los patrones delictivos post-pandemia, pero con un aumento en 2023 y 2024.

Los tres coeficientes son significativos en todos los años:

- El β de cámaras sigue un patrón en V.
- El β de aglomeraciones es el más estable entre años. 
- El β de densidad poblacional cae notoriamente en 2024 (0.195 vs ~0.32 en años anteriores).

El Moran sobre residuos es positivo y significativo en todos los años, confirmando que la heterogeneidad espacial persiste en el tiempo.

Limitación: las variables predictoras (cámaras, aglomeraciones y población) son datos que no varian entre años. Esto impide determinar si los cambios en los coeficientes se deben a variaciones en los delitos o a cambios en los predictores.

## 8. Guardar resultados finales

In [ ]:
os.makedirs('resultados', exist_ok=True)

# GeoDataFrame completo con todos los resultados
gdf.to_file('resultados/analisis_final.gpkg', driver='GPKG')
print('Guardado: resultados/analisis_final.gpkg')

# CSV sin geometría (para tablas del informe)
cols_csv = [c for c in gdf.columns if c != 'geometry']
gdf[cols_csv].to_csv('resultados/analisis_final.csv', index=False)
print('Guardado: resultados/analisis_final.csv')

print('\nArchivos generados para el informe:')
for f in sorted(os.listdir('resultados')):
    if f.endswith('.svg'):
        print(f'  📊 {f}')

for f in sorted(os.listdir('resultados')):
    if f.endswith('.png'):
        print(f'  📊 {f}')

## 9. Discusión y límites

### Resultados

Los resultados confirman que el crimen en la CDMX presenta alta concentración espacial, no se distribuye aleatoriamente. El 15.7% de las colonias concentra el 50% de los delitos (Gini=0.555).

Los hotspots son estables y concentrados en el centro y oriente, pero la concentración se debilita progresivamente desde 2019 (I=0.398) hasta 2024 (I=0.258), sugiriendo una leve dispersión del crimen hacia la periferia.

Cada categoría de delito tiene su propia geografía y horario, lo que implica que una política de prevención debe considerar qué tipo de delito ocurre, dónde y a qué hora.

El GWR revela que los efectos de los predictores no son uniformes, siendo el hallazgo más relevante que la densidad poblacional tiene efecto negativo en el centro, consistente con el argumento de población flotante.

### Límites

**Datos:** el Censo 2010 está desactualizado; la población flotante sería el denominador más adecuado para delitos de espacio público. Los datos de 2024 son parciales (enero–septiembre).

**Predictores estáticos:** cámaras y aglomeraciones son snapshots de un momento único.

**Causalidad inversa:** el coeficiente positivo de cámaras no implica que causen delitos, sino que se instalan donde ya los hay.

**Cifra negra:** los datos capturan solo los delitos denunciados (~7% del total estimado), por lo que los patrones observados reflejan denuncias, no necesariamente ocurrencia real.

## 10. Conclusiones

Sí existen patrones espaciales y temporales en la distribución de delitos de alto impacto en la CDMX, y estos varían según la categoría de delito.

**Subpregunta 1:** cada categoría tiene su propia geografía delictiva. El centro aparece como zona de alta densidad en dos de las cuatro categorías, pero los patrones no son idénticos.

**Subpregunta 2:** la hora del delito depende del tipo. El robo a repartidor ocurre principalmente entre las 10 y 14 horas, el robo a negocio tiene su peak al cierre (entre 20 y 21 horas).

**Subpregunta 3:** existen hotspots significativos concentrados en el centro y oriente (383 colonias HH), estables entre 2019 y 2024, con una disminución progresiva desde la pandemia (371 HH en 2019 vs 218 en 2024). Los coldspots dominan la periferia sur y oeste.

**Análisis explicativo:** El GWR confirma que los factores explicativos operan de forma heterogénea ($R^2:\text{GWR}=0.500\text{ vs OLS}=0.284$): la densidad poblacional cambia de signo según la zona, y el $R^2$ local bajo en el centro indica que ahí operan factores no capturados por el modelo.